# AdaptAuth-ML · Experimento base S8, versión 3 (Google Colab)

**Desde el celular:** menú **Entorno de ejecución → Ejecutar todo**. Al inicio te pedirá permiso para conectar tu Google Drive: acéptalo.

**El dataset se descarga una sola vez.** Todo se guarda en **Mi unidad → AdaptAuth-ML**:
- `rba-dataset.zip`: el dataset original de Zenodo (1.1 GB), se descarga solo la primera vez.
- `rba_muestra_usuarios_5pct.parquet`: la muestra ya filtrada (5 % de usuarios con todos sus accesos), se crea solo la primera vez.

| Corrida | Qué hace | Tiempo aprox. |
|---|---|---|
| Primera vez | Descarga el zip a Drive, crea la muestra y ejecuta | 20 a 30 min |
| Siguientes | Lee la muestra desde Drive y ejecuta | 5 a 10 min |

Si ya tienes el dataset en tu Drive (en cualquier carpeta y con cualquier nombre que contenga "rba"), el notebook lo encuentra solo y no lo vuelve a descargar.

In [ ]:
!pip -q install shap

## 1. Código del pipeline

In [ ]:
%%writefile adaptauth.py
"""
AdaptAuth-ML - pipeline del baseline S8 (Isolation Forest + reglas contextuales).

Implementa el diseño corregido del PI7 (Fase 0):
  - x_i en R^8 (hora cíclica, desviación horaria, historial, frecuencias de país/ASN/dispositivo, RTT)
  - c_i = 0.25 r1 + 0.25 r2 + 0.30 r3 + 0.15 r4 + 0.05 r5
  - s(x) de Isolation Forest (ecuación 1) y reescalado s'(x) (ecuación 1b)
  - R_i = alpha * s'(x_i) + (1 - alpha) * c_i (ecuación 2)
  - Política: acceso directo / OTP / bloqueo, con regla de arranque en frío (P5)
  - Partición temporal 70/15/15 y calibración de alpha, theta1, theta2 solo en validación

Los nombres de columnas siguen el dataset RBA (Wiefling et al., 2022), de modo que el
mismo código corre sobre el archivo real o sobre los datos sintéticos de prueba.
"""
from __future__ import annotations

import json
import time
from dataclasses import dataclass, field, asdict

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.ensemble import IsolationForest

# --------------------------------------------------------------------------------------
# Columnas del dataset RBA
# --------------------------------------------------------------------------------------
COL_TS = "Login Timestamp"
COL_USER = "User ID"
COL_RTT = "Round-Trip Time [ms]"
COL_IP = "IP Address"
COL_COUNTRY = "Country"
COL_ASN = "ASN"
COL_BROWSER = "Browser Name and Version"
COL_OS = "OS Name and Version"
COL_DEVICE = "Device Type"
COL_SUCCESS = "Login Successful"
COL_ATTACK_IP = "Is Attack IP"          # SOLO etiqueta de evaluación, nunca entrada
COL_ATO = "Is Account Takeover"         # SOLO etiqueta de evaluación, nunca entrada

LABEL_COLUMNS = (COL_ATTACK_IP, COL_ATO)

FEATURES = [
    "hora_sin", "hora_cos", "desv_horaria", "log_n_prev",
    "frec_pais", "frec_asn", "frec_dispositivo", "log_rtt",
]
RULES = ["r1_dispositivo_nuevo", "r2_pais_poco_habitual", "r3_viaje_imposible",
         "r4_horario_atipico", "r5_asn_nuevo"]
RULE_WEIGHTS = {"r1_dispositivo_nuevo": 0.25, "r2_pais_poco_habitual": 0.25,
                "r3_viaje_imposible": 0.30, "r4_horario_atipico": 0.15, "r5_asn_nuevo": 0.05}

DECISION_NAMES = {0: "acceso_directo", 1: "otp", 2: "bloqueo"}


@dataclass
class Config:
    seed: int = 42
    n_estimators: int = 100          # t
    max_samples: int = 256           # n
    split: tuple = (0.70, 0.15, 0.15)
    alpha_grid: tuple = (0.3, 0.5, 0.7, 0.85, 1.0)
    theta_grid: tuple = tuple(np.round(np.arange(0.02, 0.951, 0.01), 2))
    max_fpr_theta1: float = 0.05     # RNF02 / CA-C2-02
    max_fpr_theta2: float = 0.01     # bloqueo: tolerancia más estricta
    cold_start_otp: bool = True      # regla P5
    n_test_blocks: int = 10          # bloques temporales para la prueba pareada
    label: str = "anomalo"           # columna de verdad usada para calibrar


# --------------------------------------------------------------------------------------
# 1. Datos
# --------------------------------------------------------------------------------------
def _open_csv(path: str):
    if str(path).endswith(".zip"):          # se lee directo del .zip, sin descomprimir ~9 GB
        import zipfile
        zf = zipfile.ZipFile(path)
        member = next(n for n in zf.namelist() if n.endswith(".csv"))
        return zf.open(member)
    return path


def load_rba(path: str, nrows: int | None = None, user_pct: float | None = None,
             chunksize: int = 2_000_000) -> pd.DataFrame:
    """
    Carga el CSV real de RBA (rba-dataset.csv o el .zip de Zenodo).
      - nrows: primeras N filas (ventana de tiempo corta: muchos usuarios sin historial).
      - user_pct: muestreo POR USUARIO sobre todo el archivo (p. ej. 5 = 5 % de usuarios con
        TODOS sus accesos). Conserva historiales completos; es la opción recomendada.
    """
    usecols = [COL_TS, COL_USER, COL_RTT, COL_IP, COL_COUNTRY, COL_ASN, COL_BROWSER,
               COL_OS, COL_DEVICE, COL_SUCCESS, COL_ATTACK_IP, COL_ATO]
    if str(path).endswith(".parquet"):      # muestra ya filtrada y guardada (lectura en segundos)
        df = pd.read_parquet(path)
        df[COL_TS] = pd.to_datetime(df[COL_TS])
        return df
    if user_pct:
        parts = []
        for ch in pd.read_csv(_open_csv(path), usecols=usecols, chunksize=chunksize):
            h = pd.util.hash_pandas_object(ch[COL_USER], index=False).to_numpy() % 10000
            parts.append(ch[h < user_pct * 100])
        df = pd.concat(parts, ignore_index=True)
    else:
        df = pd.read_csv(_open_csv(path), usecols=usecols, nrows=nrows)
    df[COL_TS] = pd.to_datetime(df[COL_TS])
    for col in (COL_SUCCESS, COL_ATTACK_IP, COL_ATO):
        if df[col].dtype == object:
            df[col] = df[col].astype(str).str.lower().isin(["true", "1"])
    return df


def generate_synthetic_rba(n_users: int = 2000, days: int = 365, seed: int = 42) -> pd.DataFrame:
    """
    Datos SINTÉTICOS con la misma estructura de columnas que RBA. Sirven solo para
    probar el pipeline de punta a punta; sus métricas NO son resultados de la tesis.
    """
    rng = np.random.default_rng(seed)
    start = pd.Timestamp("2020-02-03")
    countries = np.array(["NO", "SE", "DE", "GB", "US", "FR", "ES", "PL", "BR", "IN"])
    home_p = np.array([0.55, 0.08, 0.07, 0.06, 0.06, 0.05, 0.04, 0.04, 0.03, 0.02])
    asn_pool = {c: [10000 + 100 * i + k for k in range(6)] for i, c in enumerate(countries)}
    attacker_countries = np.array(["US", "RU", "CN", "VN", "BR", "NL", "UA", "ID"])
    attacker_asns = [64000 + k for k in range(60)]       # proveedores de hosting
    browsers = ["Chrome 88", "Firefox 85", "Safari 14", "Edge 88", "Chrome Mobile 88",
                "Samsung Internet 13", "Opera 74"]
    oses = ["Windows 10", "Mac OS X 10.15", "iOS 14.4", "Android 10", "Linux", "Android 11"]
    devices = ["desktop", "mobile", "tablet"]
    base_rtt = {c: v for c, v in zip(countries, [30, 45, 60, 65, 140, 70, 90, 70, 220, 260])}

    rows = []
    for uid in range(n_users):
        home = rng.choice(countries, p=home_p)
        user_asns = rng.choice(asn_pool[home], size=rng.integers(1, 3), replace=False)
        n_dev = rng.integers(1, 4)
        user_devs = [(rng.choice(browsers), rng.choice(oses), rng.choice(devices, p=[.5, .42, .08]))
                     for _ in range(n_dev)]
        pref_hour = np.clip(rng.normal(13, 3), 6, 22)
        n_events = int(np.clip(rng.lognormal(4.2, 0.9), 2, 900))
        times = np.sort(rng.uniform(0, days, n_events))
        for t in times:
            day = np.floor(t)
            hour = (rng.normal(pref_hour, 2.5)) % 24
            ts = start + pd.Timedelta(days=float(day)) + pd.Timedelta(hours=float(hour))
            ctry, asn = home, rng.choice(user_asns)
            if rng.random() < 0.02:                       # viaje legítimo
                ctry = rng.choice(countries)
                asn = rng.choice(asn_pool[ctry])
            elif rng.random() < 0.08:                     # red móvil / otra red del país
                asn = rng.choice(asn_pool[home])
            dev = user_devs[rng.integers(len(user_devs))]
            if rng.random() < 0.03:                       # dispositivo nuevo legítimo
                dev = (rng.choice(browsers), rng.choice(oses), rng.choice(devices))
                user_devs.append(dev)
            rtt = rng.lognormal(np.log(base_rtt[ctry]), 0.35)
            rows.append((ts, uid, rtt, ctry, asn, dev[0], dev[1], dev[2],
                         rng.random() > 0.04, 0, 0))

    n_legit = len(rows)
    # Relleno de credenciales (credential stuffing) desde IPs de ataque
    n_attack = int(n_legit * 0.08)
    for _ in range(n_attack):
        uid = int(rng.integers(n_users))
        ts = start + pd.Timedelta(days=float(rng.uniform(0, days)))
        ctry = rng.choice(attacker_countries)
        rtt = rng.lognormal(np.log(180), 0.6)
        rows.append((ts, uid, rtt, ctry, rng.choice(attacker_asns), rng.choice(browsers),
                     rng.choice(oses), "desktop", rng.random() < 0.02, 1, 0))
    # Toma de cuenta (account takeover): acceso exitoso del atacante
    n_ato = int(n_legit * 0.002)
    for _ in range(n_ato):
        uid = int(rng.integers(n_users))
        ts = start + pd.Timedelta(days=float(rng.uniform(30, days)))
        ctry = rng.choice(attacker_countries)
        rtt = rng.lognormal(np.log(180), 0.6)
        rows.append((ts, uid, rtt, ctry, rng.choice(attacker_asns), rng.choice(browsers),
                     rng.choice(oses), rng.choice(devices), True, 1, 1))

    df = pd.DataFrame(rows, columns=[COL_TS, COL_USER, COL_RTT, COL_COUNTRY, COL_ASN,
                                     COL_BROWSER, COL_OS, COL_DEVICE, COL_SUCCESS,
                                     COL_ATTACK_IP, COL_ATO])
    df[COL_IP] = "sintetico"
    return df


# --------------------------------------------------------------------------------------
# 2. C1 - Ingesta y feature engineering (solo usa el pasado de cada usuario)
# --------------------------------------------------------------------------------------
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Construye x_i (R^8), las reglas r1..r5 y c_i. El historial de cada usuario se forma
    SOLO con sus inicios de sesión exitosos anteriores al evento, de modo que no hay
    fuga temporal ni se usan las etiquetas de ataque como entrada.
    """
    d = df.sort_values(COL_TS, kind="mergesort").reset_index(drop=True).copy()
    d["anomalo"] = ((d[COL_ATTACK_IP] == 1) | (d[COL_ATO] == 1)).astype(int)
    d["dispositivo"] = (d[COL_BROWSER].astype(str) + "|" + d[COL_OS].astype(str) + "|"
                        + d[COL_DEVICE].astype(str))
    succ = d[COL_SUCCESS].astype(bool).astype(int)
    d["_succ"] = succ

    hour = d[COL_TS].dt.hour + d[COL_TS].dt.minute / 60.0
    ang = 2 * np.pi * hour / 24.0
    d["hora_sin"], d["hora_cos"] = np.sin(ang), np.cos(ang)

    g = d.groupby(COL_USER)
    n_prev = g["_succ"].cumsum() - succ                   # accesos exitosos previos
    d["n_prev"] = n_prev
    d["log_n_prev"] = np.log1p(n_prev)

    def prior_share(key: str) -> pd.Series:
        same = d.groupby([COL_USER, key])["_succ"].cumsum() - succ
        return np.where(n_prev > 0, same / n_prev.clip(lower=1), 0.0)

    d["frec_pais"] = prior_share(COL_COUNTRY)
    d["frec_asn"] = prior_share(COL_ASN)
    d["frec_dispositivo"] = prior_share("dispositivo")

    # Hora habitual: media circular de los accesos exitosos previos
    s_cum = (d["hora_sin"] * succ).groupby(d[COL_USER]).cumsum() - d["hora_sin"] * succ
    c_cum = (d["hora_cos"] * succ).groupby(d[COL_USER]).cumsum() - d["hora_cos"] * succ
    mean_ang = np.arctan2(s_cum, c_cum)
    diff = np.abs(np.angle(np.exp(1j * (ang - mean_ang))))     # 0..pi
    desv_h = diff * 24 / (2 * np.pi)                            # 0..12 h
    d["desv_horas"] = np.where(n_prev > 0, desv_h, np.nan)
    d["desv_horaria"] = np.where(n_prev > 0, desv_h / 12.0, 0.5)

    d["log_rtt"] = np.log1p(d[COL_RTT].astype(float).fillna(d[COL_RTT].median()))

    # Último acceso exitoso previo (para viaje imposible)
    d["_ctry_s"] = d[COL_COUNTRY].where(succ == 1)
    d["_ts_s"] = d[COL_TS].where(succ == 1)
    # Vectorizado (sin lambda por usuario) para soportar millones de filas
    d["prev_pais"] = d.groupby(COL_USER)["_ctry_s"].shift(1).groupby(d[COL_USER]).ffill()
    d["prev_ts"] = d.groupby(COL_USER)["_ts_s"].shift(1).groupby(d[COL_USER]).ffill()
    horas_desde_prev = (d[COL_TS] - d["prev_ts"]).dt.total_seconds() / 3600.0

    hist = n_prev >= 1
    d["r1_dispositivo_nuevo"] = (hist & (d["frec_dispositivo"] == 0)).astype(int)
    d["r2_pais_poco_habitual"] = (hist & (d["frec_pais"] < 0.2)).astype(int)
    d["r3_viaje_imposible"] = (d["prev_pais"].notna() & (d["prev_pais"] != d[COL_COUNTRY])
                               & (horas_desde_prev < 2)).astype(int)
    d["r4_horario_atipico"] = ((n_prev >= 5) & (d["desv_horas"] > 4)).astype(int)
    d["r5_asn_nuevo"] = (hist & (d["frec_asn"] == 0)).astype(int)
    d["c_i"] = sum(RULE_WEIGHTS[r] * d[r] for r in RULES)
    d["arranque_frio"] = (n_prev == 0).astype(int)

    return d.drop(columns=["_succ", "_ctry_s", "_ts_s"])


def diagnose(raw: pd.DataFrame, d: pd.DataFrame) -> dict:
    """
    Análisis exploratorio orientado a la detección: ¿qué variables separan de verdad los
    accesos anómalos de los legítimos? (Etapa 1 del procedimiento metodológico).
    """
    from sklearn.metrics import roc_auc_score
    y = d["anomalo"].to_numpy()
    lab = np.where(y == 1, "anomalo", "legitimo")
    per_user = d.groupby(COL_USER).size()

    def auc(col, mask=None):
        m = np.ones(len(d), bool) if mask is None else mask
        yy, xx = y[m], d.loc[m, col].astype(float).fillna(0).to_numpy()
        return round(float(roc_auc_score(yy, xx)), 3) if 0 < yy.sum() < len(yy) else None

    hist = (d["n_prev"] >= 1).to_numpy()
    cols = FEATURES + RULES + ["c_i", COL_RTT]

    def top_share(col, k=5):
        out = {}
        for name in ("legitimo", "anomalo"):
            s = d.loc[lab == name, col].astype(str).value_counts(normalize=True).head(k)
            out[name] = {i: round(float(v), 3) for i, v in s.items()}
        return out

    legit_asn = set(d.loc[y == 0, COL_ASN])
    legit_users = set(d.loc[(y == 0) & d[COL_SUCCESS].astype(bool), COL_USER])
    atk = d[y == 1]
    return {
        "archivo_ordenado_por_fecha": bool(raw[COL_TS].is_monotonic_increasing),
        "rango_fechas": [str(d[COL_TS].min()), str(d[COL_TS].max())],
        "eventos": int(len(d)), "usuarios": int(per_user.size),
        "eventos_por_usuario_p50_p90_max": [int(per_user.quantile(.5)), int(per_user.quantile(.9)),
                                            int(per_user.max())],
        "tasa_anomalos": round(float(y.mean()), 4), "n_ato": int(d[COL_ATO].sum()),
        "exito_login": {k: round(float(d.loc[lab == k, COL_SUCCESS].astype(bool).mean()), 4)
                        for k in ("legitimo", "anomalo")},
        "primer_acceso": {k: round(float((d.loc[lab == k, "n_prev"] == 0).mean()), 4)
                          for k in ("legitimo", "anomalo")},
        "anomalos_sobre_usuarios_con_historial_legitimo": round(float(atk[COL_USER].isin(legit_users).mean()), 4),
        "anomalos_con_asn_visto_en_legitimos": round(float(atk[COL_ASN].isin(legit_asn).mean()), 4),
        "pais_top5": top_share(COL_COUNTRY), "tipo_dispositivo_top5": top_share(COL_DEVICE),
        "asn_top5": top_share(COL_ASN),
        "auc_univariada_todos": {c: auc(c) for c in cols},
        "auc_univariada_con_historial": {c: auc(c, hist) for c in cols},
        "tasa_reglas": {r: {k: round(float(d.loc[lab == k, r].mean()), 4) for k in ("legitimo", "anomalo")}
                        for r in RULES},
        "nota_auc": "0.5 = no separa; > 0.5 = valores altos indican anomalía; < 0.5 = relación invertida",
    }


def temporal_split(d: pd.DataFrame, split=(0.70, 0.15, 0.15)):
    n = len(d)
    a, b = int(n * split[0]), int(n * (split[0] + split[1]))
    return d.iloc[:a].copy(), d.iloc[a:b].copy(), d.iloc[b:].copy()


# --------------------------------------------------------------------------------------
# 3. C2 / C3 / C4 - detección, consolidación y decisión
# --------------------------------------------------------------------------------------
def train_iforest(train: pd.DataFrame, cfg: Config) -> IsolationForest:
    model = IsolationForest(n_estimators=cfg.n_estimators, max_samples=cfg.max_samples,
                            contamination="auto", random_state=cfg.seed, n_jobs=-1)
    model.fit(train[FEATURES].to_numpy())
    return model


def anomaly_score(model: IsolationForest, d: pd.DataFrame) -> np.ndarray:
    """Ecuación 1: s(x, n) = 2^(-E[h(x)]/c(n)). En scikit-learn, score_samples = -s(x, n)."""
    return -model.score_samples(d[FEATURES].to_numpy())


def rescale(s: np.ndarray, s_ref: float) -> np.ndarray:
    """Ecuación 1b: s'(x) = max(0, (s - s_ref) / (1 - s_ref))."""
    return np.clip((s - s_ref) / (1.0 - s_ref), 0.0, 1.0)


def risk_score(s_prime: np.ndarray, c: np.ndarray, alpha: float) -> np.ndarray:
    """Ecuación 2: R_i = alpha * s'(x_i) + (1 - alpha) * c_i."""
    return alpha * s_prime + (1.0 - alpha) * c


def decide(R: np.ndarray, theta1: float, theta2: float, cold: np.ndarray | None = None,
           cold_start_otp: bool = True) -> np.ndarray:
    """Reglas P2, P3 y P5. 0 = acceso directo, 1 = OTP, 2 = bloqueo."""
    dec = np.where(R > theta2, 2, np.where(R > theta1, 1, 0))
    if cold_start_otp and cold is not None:
        dec = np.where((cold == 1) & (dec == 0), 1, dec)
    return dec


# --------------------------------------------------------------------------------------
# 4. Métricas (dos análisis binarios sobre la decisión de tres niveles)
# --------------------------------------------------------------------------------------
def binary_metrics(y: np.ndarray, pred: np.ndarray) -> dict:
    tp = int(((pred == 1) & (y == 1)).sum()); fp = int(((pred == 1) & (y == 0)).sum())
    fn = int(((pred == 0) & (y == 1)).sum()); tn = int(((pred == 0) & (y == 0)).sum())
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    fpr = fp / (fp + tn) if fp + tn else 0.0
    fnr = fn / (fn + tp) if fn + tp else 0.0
    denom = np.sqrt(float(tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    mcc = ((tp * tn) - (fp * fn)) / denom if denom else 0.0
    return dict(tp=tp, fp=fp, fn=fn, tn=tn, precision=precision, recall=recall, f1=f1,
                fpr=fpr, fnr=fnr, mcc=mcc)


def evaluate_decisions(y: np.ndarray, dec: np.ndarray) -> dict:
    """Análisis theta1: positivo = OTP o bloqueo. Análisis theta2: positivo = bloqueo."""
    return {"analisis_theta1": binary_metrics(y, (dec >= 1).astype(int)),
            "analisis_theta2": binary_metrics(y, (dec == 2).astype(int))}


def confusion_3x2(y: np.ndarray, dec: np.ndarray) -> pd.DataFrame:
    tab = pd.crosstab(pd.Series(dec).map(DECISION_NAMES).rename("decision"),
                      pd.Series(y).map({0: "legitimo", 1: "anomalo"}).rename("etiqueta_real"))
    return tab.reindex(index=list(DECISION_NAMES.values()), columns=["legitimo", "anomalo"],
                       fill_value=0)


# --------------------------------------------------------------------------------------
# 5. Calibración en validación (el conjunto de prueba no se usa aquí)
# --------------------------------------------------------------------------------------
def calibrate(val: pd.DataFrame, s_prime: np.ndarray, cfg: Config, alpha_grid) -> dict:
    """
    Búsqueda en malla: maximiza F1 del análisis theta1 sujeto a FPR <= max_fpr_theta1.
    theta2 es el menor umbral > theta1 cuyo FPR de bloqueo <= max_fpr_theta2.
    """
    y = val[cfg.label].to_numpy()
    cold = val["arranque_frio"].to_numpy()
    c = val["c_i"].to_numpy()
    best, fallback = None, None
    for alpha in alpha_grid:
        R = risk_score(s_prime, c, alpha)
        for t1 in cfg.theta_grid:
            dec = decide(R, t1, 1.01, cold, cfg.cold_start_otp)
            m = binary_metrics(y, (dec >= 1).astype(int))
            cand = dict(alpha=float(alpha), theta1=float(t1), f1_val=m["f1"], fpr_val=m["fpr"])
            if m["fpr"] <= cfg.max_fpr_theta1 and (best is None or m["f1"] > best["f1_val"]):
                best = cand
            # respaldo: menor FPR posible y, a igualdad, mayor F1
            if fallback is None or (m["fpr"], -m["f1"]) < (fallback["fpr_val"], -fallback["f1_val"]):
                fallback = cand
    if best is None:   # ninguna combinación cumple FPR <= umbral: se declara y se usa el respaldo
        best = dict(fallback, restriccion_fpr_incumplida=True)
    R = risk_score(s_prime, c, best["alpha"])
    best["theta2"] = 0.95
    for t2 in cfg.theta_grid:
        if t2 <= best["theta1"]:
            continue
        m = binary_metrics(y, (R > t2).astype(int))
        if m["fpr"] <= cfg.max_fpr_theta2:
            best["theta2"] = float(t2)
            break
    return best


# --------------------------------------------------------------------------------------
# 6. Prueba pareada por bloques temporales (baseline vs AdaptAuth-ML)
# --------------------------------------------------------------------------------------
def paired_block_test(test: pd.DataFrame, dec_a: np.ndarray, dec_b: np.ndarray,
                      y_col: str, n_blocks: int) -> dict:
    """
    Divide el conjunto de prueba en n bloques temporales consecutivos y compara el F1
    (análisis theta1) de ambas condiciones sobre los MISMOS bloques: muestras pareadas.
    """
    y = test[y_col].to_numpy()
    idx = np.array_split(np.arange(len(test)), n_blocks)
    f1_a = np.array([binary_metrics(y[i], (dec_a[i] >= 1).astype(int))["f1"] for i in idx])
    f1_b = np.array([binary_metrics(y[i], (dec_b[i] >= 1).astype(int))["f1"] for i in idx])
    diff = f1_b - f1_a
    shapiro_p = float(stats.shapiro(diff).pvalue) if np.ptp(diff) > 0 else 1.0
    if shapiro_p > 0.05:
        t = stats.ttest_rel(f1_b, f1_a)
        test_name, stat, p = "t de Student pareada", float(t.statistic), float(t.pvalue)
    else:
        w = stats.wilcoxon(f1_b, f1_a)
        test_name, stat, p = "Wilcoxon de rangos con signo", float(w.statistic), float(w.pvalue)
    sd = diff.std(ddof=1)
    dz = float(diff.mean() / sd) if sd > 0 else float("inf")
    return dict(n_bloques=n_blocks, f1_baseline=f1_a.tolist(), f1_adaptauth=f1_b.tolist(),
                media_baseline=float(f1_a.mean()), media_adaptauth=float(f1_b.mean()),
                de_baseline=float(f1_a.std(ddof=1)), de_adaptauth=float(f1_b.std(ddof=1)),
                shapiro_p_diferencias=shapiro_p, prueba=test_name, estadistico=stat, p_valor=p,
                d_cohen_dz=dz)


# --------------------------------------------------------------------------------------
# 7. Latencia (RNF03 / CA-C2-03)
# --------------------------------------------------------------------------------------
def inference_latency_ms(model: IsolationForest, d: pd.DataFrame, n: int = 300, seed: int = 0) -> dict:
    rng = np.random.default_rng(seed)
    X = d[FEATURES].to_numpy()
    times = []
    for i in rng.integers(0, len(X), n):
        t0 = time.perf_counter()
        model.score_samples(X[i:i + 1])
        times.append((time.perf_counter() - t0) * 1000)
    t = np.array(times)
    return dict(p50_ms=float(np.percentile(t, 50)), p95_ms=float(np.percentile(t, 95)),
                p99_ms=float(np.percentile(t, 99)), media_ms=float(t.mean()))


def to_json(obj, path):
    def conv(o):
        if isinstance(o, (np.integer,)):
            return int(o)
        if isinstance(o, (np.floating,)):
            return float(o)
        if isinstance(o, Config):
            return asdict(o)
        raise TypeError(type(o))
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, default=conv)


In [ ]:
%%writefile figures.py
"""Figuras del experimento base (paleta de referencia: azul y naranja, marcas finas)."""
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

BLUE, ORANGE = "#2a78d6", "#eb6834"
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e4e3df"

plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 11, "axes.edgecolor": GRID,
    "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 150,
})


def _grid(ax, axis="y"):
    ax.grid(axis=axis, color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)


def risk_distribution(R, y, t1, t2, path):
    fig, ax = plt.subplots(figsize=(8, 4.2))
    bins = np.linspace(0, 1, 41)
    ax.hist(R[y == 0], bins=bins, density=True, histtype="step", linewidth=2, color=BLUE,
            label="Accesos legítimos")
    ax.hist(R[y == 1], bins=bins, density=True, histtype="step", linewidth=2, color=ORANGE,
            label="Accesos anómalos")
    ax.set_yscale("log")
    top = ax.get_ylim()[1]
    for t, name, f, ha, dx in ((t1, "θ1", 0.5, "right", -0.01), (t2, "θ2", 0.12, "left", 0.01)):
        ax.axvline(t, color=INK2, linewidth=1, linestyle="--")
        ax.text(t + dx, top * f, f"{name} = {t:.2f}", color=INK2, fontsize=10, ha=ha)
    ax.set_xlabel("Score de riesgo consolidado R_i")
    ax.set_ylabel("Densidad (escala log)")
    _grid(ax)
    ax.legend(frameon=False, loc="upper right")
    fig.tight_layout(); fig.savefig(path); plt.close(fig)


def paired_blocks(res, path):
    a, b = np.array(res["f1_baseline"]), np.array(res["f1_adaptauth"])
    x = np.arange(1, len(a) + 1)
    fig, ax = plt.subplots(figsize=(8, 4.2))
    ax.plot(x, a, color=BLUE, linewidth=2, marker="o", markersize=7, label="Baseline (solo reglas)")
    ax.plot(x, b, color=ORANGE, linewidth=2, marker="o", markersize=7, label="AdaptAuth-ML")
    ax.set_xticks(x)
    ax.set_xlabel("Bloque temporal del conjunto de prueba")
    ax.set_ylabel("F1 (análisis θ1)")
    ax.set_ylim(0, 1)
    _grid(ax)
    ax.legend(frameon=False, loc="lower right")
    fig.tight_layout(); fig.savefig(path); plt.close(fig)


def shap_bars(mean_abs, path):
    s = mean_abs.sort_values()
    fig, ax = plt.subplots(figsize=(8, 4.2))
    ax.barh(s.index, s.values, color=BLUE, height=0.6)
    for i, v in enumerate(s.values):
        ax.text(v, i, f" {v:.3f}", va="center", color=INK2, fontsize=9)
    ax.set_xlabel("Importancia media |SHAP| sobre el score de anomalía")
    _grid(ax, "x")
    fig.tight_layout(); fig.savefig(path); plt.close(fig)


In [ ]:
%%writefile run_baseline.py
"""
Ejecuta el experimento base de la Semana 8 de AdaptAuth-ML.

Uso:
    python run_baseline.py --data data/rba-dataset.csv [--nrows 2000000]   # datos reales RBA
    python run_baseline.py --synthetic                                     # datos de prueba

Salidas en results/: metricas.json, tablas .csv y figuras .png.
"""
import argparse
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path(__file__).parent / "src"))
import adaptauth as aa  # noqa: E402
import figures  # noqa: E402

OUT = Path(__file__).parent / "results"


def main():
    ap = argparse.ArgumentParser()
    g = ap.add_mutually_exclusive_group(required=True)
    g.add_argument("--data", help="ruta al CSV real de RBA")
    g.add_argument("--synthetic", action="store_true", help="usar datos sintéticos de prueba")
    ap.add_argument("--nrows", type=int, default=None)
    ap.add_argument("--user-pct", type=float, default=None,
                    help="muestreo por usuario sobre todo el archivo (p. ej. 5 = 5 %% de usuarios)")
    ap.add_argument("--no-shap", action="store_true")
    args = ap.parse_args()

    cfg = aa.Config()
    OUT.mkdir(exist_ok=True); (OUT / "figuras").mkdir(exist_ok=True)
    origen = ("SINTETICO (solo prueba del pipeline)" if args.synthetic else
              f"RBA real: {args.data} | " + (f"muestreo por usuario {args.user_pct} %" if args.user_pct
                                            else f"primeras {args.nrows} filas"))

    t0 = time.time()
    raw = aa.generate_synthetic_rba(seed=cfg.seed) if args.synthetic else aa.load_rba(args.data, args.nrows, args.user_pct)
    d = aa.build_features(raw)
    diag = aa.diagnose(raw, d)
    aa.to_json(diag, OUT / "diagnostico.json")
    print("[EDA] ordenado por fecha:", diag["archivo_ordenado_por_fecha"], "| rango:", diag["rango_fechas"],
          "| usuarios:", f"{diag['usuarios']:,}", "| anomalos:", diag["tasa_anomalos"])
    train, val, test = aa.temporal_split(d, cfg.split)
    print(f"[C1] {len(d):,} eventos | train {len(train):,} | val {len(val):,} | test {len(test):,}"
          f" | {time.time() - t0:.1f}s")
    for name, part in (("train", train), ("val", val), ("test", test)):
        print(f"     {name}: anomalos={int(part['anomalo'].sum()):,} | ATO={int(part[aa.COL_ATO].sum()):,}"
              f" | primeros accesos={int(part['arranque_frio'].sum()):,}")
    if val["anomalo"].sum() == 0 or test["anomalo"].sum() == 0:
        sys.exit("ERROR: validación o prueba no tienen eventos anómalos; aumenta --nrows.")

    # C2: Isolation Forest (no supervisado: no ve etiquetas)
    model = aa.train_iforest(train, cfg)
    s_val, s_test = aa.anomaly_score(model, val), aa.anomaly_score(model, test)

    # Ecuación 1b: s_ref = mediana de s en accesos legítimos de VALIDACIÓN
    s_ref = float(np.median(s_val[val[cfg.label].to_numpy() == 0]))
    sp_val, sp_test = aa.rescale(s_val, s_ref), aa.rescale(s_test, s_ref)

    # Calibración solo en validación
    cal_adapt = aa.calibrate(val, sp_val, cfg, cfg.alpha_grid)          # AdaptAuth-ML
    cal_base = aa.calibrate(val, sp_val, cfg, (0.0,))                   # baseline: solo reglas
    print(f"[C3/C4] s_ref={s_ref:.3f} | AdaptAuth {cal_adapt} | Baseline {cal_base}")

    y = test[cfg.label].to_numpy(); y_ato = test[aa.COL_ATO].to_numpy()
    cold = test["arranque_frio"].to_numpy(); c = test["c_i"].to_numpy()
    R_adapt = aa.risk_score(sp_test, c, cal_adapt["alpha"])
    R_base = aa.risk_score(sp_test, c, 0.0)
    dec_adapt = aa.decide(R_adapt, cal_adapt["theta1"], cal_adapt["theta2"], cold, cfg.cold_start_otp)
    dec_base = aa.decide(R_base, cal_base["theta1"], cal_base["theta2"], cold, cfg.cold_start_otp)
    # Isolation Forest solo (sin reglas ni arranque en frío), como referencia técnica
    cal_if = aa.calibrate(val.assign(c_i=0.0, arranque_frio=0), sp_val,
                          aa.Config(cold_start_otp=False), (1.0,))
    dec_if = aa.decide(sp_test, cal_if["theta1"], cal_if["theta2"], None, False)

    legit = y == 0
    resultados = {
        "origen_datos": origen,
        "advertencia": ("Datos sintéticos: estas cifras validan que el pipeline funciona, "
                        "NO son resultados de la tesis.") if args.synthetic else None,
        "config": cfg,
        "volumen": dict(total=len(d), train=len(train), val=len(val), test=len(test),
                        anomalos_test=int(y.sum()), ato_test=int(y_ato.sum()),
                        arranque_frio_test=int(cold.sum())),
        "s_ref": s_ref,
        "calibracion": dict(adaptauth=cal_adapt, baseline_reglas=cal_base, iforest_solo=cal_if),
        "prueba": {
            "adaptauth": aa.evaluate_decisions(y, dec_adapt),
            "baseline_reglas": aa.evaluate_decisions(y, dec_base),
            "iforest_solo": aa.evaluate_decisions(y, dec_if),
            # Misma evaluación excluyendo primeros accesos (regla P5 de arranque en frío)
            "adaptauth_sin_arranque_frio": aa.evaluate_decisions(y[cold == 0], dec_adapt[cold == 0]),
            "adaptauth_recall_ato":aa.binary_metrics(y_ato, (dec_adapt >= 1).astype(int))["recall"],
            "auc_roc_s": float(__import__("sklearn.metrics", fromlist=["roc_auc_score"]).roc_auc_score(y, s_test)),
            "auc_pr_s": float(__import__("sklearn.metrics", fromlist=["average_precision_score"]).average_precision_score(y, s_test)),
            "auc_roc_R": float(__import__("sklearn.metrics", fromlist=["roc_auc_score"]).roc_auc_score(y, R_adapt)),
            "score_medio_legitimos_R": float(R_adapt[legit].mean()),       # CA-C3-02
            "R_fuera_de_rango": int(((R_adapt < 0) | (R_adapt > 1)).sum()),  # CA-C3-01
        },
        "prueba_pareada": aa.paired_block_test(test, dec_base, dec_adapt, cfg.label, cfg.n_test_blocks),
        "latencia_inferencia": aa.inference_latency_ms(model, test),
    }

    aa.confusion_3x2(y, dec_adapt).to_csv(OUT / "matriz_3x2_adaptauth.csv")
    aa.confusion_3x2(y, dec_base).to_csv(OUT / "matriz_3x2_baseline.csv")
    pd.Series(dec_adapt).map(aa.DECISION_NAMES).value_counts().to_csv(OUT / "distribucion_decisiones.csv")

    # OE1: importancia de variables con SHAP por bloque temporal (estabilidad del top-3)
    if not args.no_shap:
        import shap
        explainer = shap.TreeExplainer(model)
        blocks = np.array_split(np.arange(len(test)), 5)
        rng = np.random.default_rng(cfg.seed)
        imp = []
        for b in blocks:
            sel = rng.choice(b, size=min(400, len(b)), replace=False)
            sv = explainer.shap_values(test.iloc[sel][aa.FEATURES].to_numpy())
            imp.append(np.abs(sv).mean(axis=0))
        imp = pd.DataFrame(imp, columns=aa.FEATURES, index=[f"bloque_{i+1}" for i in range(5)])
        imp.to_csv(OUT / "shap_por_bloque.csv")
        top3 = [tuple(r.sort_values(ascending=False).index[:3]) for _, r in imp.iterrows()]
        ref = set(imp.mean().sort_values(ascending=False).index[:3])
        coincid = sum(set(t) == ref for t in top3)
        resultados["shap"] = dict(top3_global=list(imp.mean().sort_values(ascending=False).index[:3]),
                                  bloques_que_coinciden=coincid, criterio="top-3 coincide en >= 4 de 5 bloques",
                                  cumple=coincid >= 4, media_abs=imp.mean().to_dict())
        figures.shap_bars(imp.mean(), OUT / "figuras" / "shap_importancia.png")

    figures.risk_distribution(R_adapt, y, cal_adapt["theta1"], cal_adapt["theta2"],
                              OUT / "figuras" / "distribucion_riesgo.png")
    figures.paired_blocks(resultados["prueba_pareada"], OUT / "figuras" / "f1_por_bloque.png")

    aa.to_json(resultados, OUT / "metricas.json")
    m = resultados["prueba"]["adaptauth"]
    print(f"[Prueba] AdaptAuth theta1: F1={m['analisis_theta1']['f1']:.3f} "
          f"FPR={m['analisis_theta1']['fpr']:.3f} | theta2: F1={m['analisis_theta2']['f1']:.3f} "
          f"FPR={m['analisis_theta2']['fpr']:.4f}")
    print(f"[Listo] {time.time() - t0:.1f}s -> {OUT}")


if __name__ == "__main__":
    main()


## 2. Dataset desde Google Drive (se descarga solo la primera vez)

In [ ]:
USER_PCT = 5      # % de usuarios. Si cambias este valor se crea una muestra nueva.
RUTA_ZIP = ""     # opcional: ruta exacta del zip en tu Drive. Si la dejas vacía, se busca sola.

import os
from google.colab import drive
drive.mount("/content/drive")
RAIZ = "/content/drive/MyDrive"
CARPETA = f"{RAIZ}/AdaptAuth-ML"
os.makedirs(CARPETA, exist_ok=True)
MUESTRA = f"{CARPETA}/rba_muestra_usuarios_{USER_PCT}pct.parquet"

if os.path.exists(MUESTRA):
    print("Muestra encontrada en Drive, no se lee el zip:", MUESTRA)
else:
    if not RUTA_ZIP:   # busca en todo tu Drive cualquier .zip o .csv con "rba" en el nombre
        candidatos = [os.path.join(r, f) for r, _, fs in os.walk(RAIZ) for f in fs
                      if "rba" in f.lower() and f.lower().endswith((".zip", ".csv"))]
        for c in candidatos:
            print(f"  encontrado: {c} ({os.path.getsize(c) / 1e9:.2f} GB)")
        if candidatos:
            RUTA_ZIP = max(candidatos, key=os.path.getsize)
        else:
            print("No está en tu Drive: descargando de Zenodo (solo esta vez)...")
            RUTA_ZIP = f"{CARPETA}/rba-dataset.zip"
            !wget -q --show-progress -O "{RUTA_ZIP}" "https://zenodo.org/records/6782156/files/rba-dataset.zip?download=1"
    print("Usando:", RUTA_ZIP)
    print("Creando la muestra por usuario (solo esta vez, tarda unos minutos)...")
    import adaptauth as aa
    aa.load_rba(RUTA_ZIP, user_pct=USER_PCT).to_parquet(MUESTRA, index=False)
    print("Muestra guardada en Drive:", MUESTRA)
!ls -lh "{CARPETA}"

## 3. Ejecutar diagnóstico y experimento

In [ ]:
!python run_baseline.py --data "{MUESTRA}"

## 4. Diagnóstico de variables

In [ ]:
import json, pandas as pd
from IPython.display import Image, display
dg = json.load(open("results/diagnostico.json"))
for k in ["archivo_ordenado_por_fecha", "rango_fechas", "eventos", "usuarios", "eventos_por_usuario_p50_p90_max",
          "tasa_anomalos", "n_ato", "exito_login", "primer_acceso",
          "anomalos_sobre_usuarios_con_historial_legitimo", "anomalos_con_asn_visto_en_legitimos"]:
    print(f"{k}: {dg[k]}")
print("\nAUC univariada (0.5 = no separa):")
display(pd.DataFrame({"todos": dg["auc_univariada_todos"], "con_historial": dg["auc_univariada_con_historial"]}))
display(pd.DataFrame(dg["tasa_reglas"]).T)
for k in ["pais_top5", "tipo_dispositivo_top5", "asn_top5"]:
    print(k, dg[k])

## 5. Resultados del experimento

In [ ]:
r = json.load(open("results/metricas.json"))
print("Origen:", r["origen_datos"]); print("Volumen:", r["volumen"]); print("s_ref:", round(r["s_ref"], 3))
print("Calibración AdaptAuth-ML:", r["calibracion"]["adaptauth"])
print("Calibración baseline:", r["calibracion"]["baseline_reglas"])
filas = {}
for nombre in ["adaptauth", "baseline_reglas", "iforest_solo", "adaptauth_sin_arranque_frio"]:
    for analisis, m in r["prueba"][nombre].items():
        filas[(nombre, analisis)] = {k: m[k] for k in ["precision", "recall", "f1", "fpr", "fnr", "mcc"]}
display(pd.DataFrame(filas).T.round(4))
print({k: round(r["prueba"][k], 4) for k in ["auc_roc_s", "auc_pr_s", "auc_roc_R", "score_medio_legitimos_R", "adaptauth_recall_ato"]})
p = r["prueba_pareada"]
print(f"Prueba pareada: {p['prueba']}, p={p['p_valor']:.4f}, dz={p['d_cohen_dz']:.2f} | F1 baseline M={p['media_baseline']:.3f} | F1 AdaptAuth M={p['media_adaptauth']:.3f}")
print("Latencia (ms):", {k: round(v, 2) for k, v in r["latencia_inferencia"].items()})
print("SHAP:", r.get("shap", {}).get("top3_global"), "| estable:", r.get("shap", {}).get("cumple"))
print(pd.read_csv("results/matriz_3x2_adaptauth.csv", index_col=0))
for f in ["distribucion_riesgo", "f1_por_bloque", "shap_importancia"]:
    display(Image(f"results/figuras/{f}.png"))

## 6. Guardar resultados en Google Drive
Quedan en **Mi unidad → AdaptAuth-ML → results_v3**. Mándame `diagnostico.json` y `metricas.json`.

In [ ]:
!rm -rf "{CARPETA}/results_v3" && cp -r results "{CARPETA}/results_v3"
print("Resultados guardados en Mi unidad / AdaptAuth-ML / results_v3")